# Clean-Slate — a quantitative teardown 🔬
### Causal residualisation · residual-WML CAPM alpha (HAC) · residual-vs-total crash · the defence stack

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Cleaner than total?: Confirmed](https://img.shields.io/badge/Cleaner_than_total%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §3.7 residual momentum (Blitz, Huij & Martens 2011): 12-1 momentum on factor-residual returns. We use a 1-factor (market) residual — a stated simplification — prove the engine on a synthetic panel, then compare to total momentum on the real S&P 500.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from clean_slate import data, momentum, strategy, decompose, extension

# Offline synthetic panel: a MOMENTUM tape (persistent idiosyncratic drift -> the *residual* carries the
# momentum) and a no-momentum NULL. The real S&P 500 verdict is in ../docs/results.md.
panel,  market,  truth = data.synthetic_panel(mom_strength=0.0016, seed=25)   # the momentum tape
panel0, market0, _     = data.synthetic_panel(mom_strength=0.0,    seed=25)   # the null
print(f"{truth.n_stocks} stocks x {truth.n_bars} days | baked mom_strength={truth.mom_strength} | null=0")


150 stocks x 4032 days | baked mom_strength=0.0016 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — is residual momentum real? | 🟡 `WEAK` | Strong on the control (alpha HAC *t* ≈ 16); on the modern S&P 500 the residual-WML alpha is **+6.2%/yr** (*t* = **+1.2**) — slightly above total momentum (+4.4%/t+0.9) but still insignificant. |
| **Tradability** | 🟡 `FRAGILE` | Thin standalone Sharpe (**+0.08**), fast turnover (**12×/yr**), short-the-losers. |
| **Cleaner than total?** | ⚪ `Confirmed` | Better skew (**-0.08** vs **+0.18**); a higher alpha; and stacked with vol-management the drawdown drops **-61% → -25%**. |

> **In one sentence:** residualising momentum gives a slightly stronger, better-skewed premium and a clean platform for crash management — but on a 1-factor residual the standalone crash reduction is incremental; the win is the stack.

*(This notebook executes on synthetic panels; the real S&P numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Residualise with a trailing (causal) regression: $\hat\varepsilon_{i,t} = r_{i,t} - \hat\beta_{i,t-1}\,r_{\text{mkt},t}$, $\hat\beta$ a rolling slope. Score $m_i = \prod_{t-252}^{t-21}(1+\hat\varepsilon_i)-1$; WML = top-minus-bottom residual decile. The synthetic bakes the persistent drift into $\theta$ (the residual); the market carries beta dispersion. The source uses FF3 residuals — we use 1-factor (market).

In [2]:
rr = momentum.residual_returns(panel, market)
print(f"residual market-correlation {rr.mean(axis=1).corr(market):+.2f} "
      f"vs raw {panel.mean(axis=1).corr(market):+.2f} -- the residual strips most of the market")

residual market-correlation +0.00 vs raw +1.00 -- the residual strips most of the market


## Beat 2 · So what?

Momentum's crash is concentrated in its *systematic* exposure: after a market crash, the winners are low-beta defensives and the losers high-beta cyclicals, so a momentum book ends up implicitly short the market right before it rebounds (Daniel–Moskowitz 2016). Residualising removes exactly that bet. The open questions: how much premium survives, how much crash is shed by a 1-factor residual, and whether stacking vol-management finishes the job. Beats 4–6 answer all three.

## Beat 3 · Pre-registered protocol

1. **Residual premium** (`decompose.capm_alpha`): residual-WML alpha + HAC *t*.
2. **Cleaner?** (`decompose.crash_comparison`): residual vs total skew/worst-month/drawdown.
3. **Stack** (`extension.defence_stack`): total → residual → residual + vol-managed.
4. **Null:** no persistence ⇒ no premium.

**Confirmed line:** residual keeps the premium with a better skew, and the stack tames the drawdown.

## Beat 4 · The teardown

### 4a · Residual alpha, momentum vs null

In [3]:
for label, (p, mk) in [('momentum', (panel, market)), ('null', (panel0, market0))]:
    a = decompose.capm_alpha(p, mk, cost_bps=5.0)
    print(f"{label:9s}: residual alpha {a['alpha_ann_pct']:+.1f}%/yr (HAC t {a['alpha_t']:+.1f}), "
          f"beta {a['beta']:+.2f}, Sharpe {a['sharpe']:+.2f}, skew {a['skew']:+.2f}")

momentum : residual alpha +32.2%/yr (HAC t +15.8), beta -0.01, Sharpe +4.43, skew +0.01


null     : residual alpha -4.2%/yr (HAC t -2.1), beta +0.02, Sharpe -0.53, skew -0.06


### 4b · Residual vs total — the crash comparison

In [4]:
cc = decompose.crash_comparison(panel, market, cost_bps=5.0)
display(pd.DataFrame(cc).T.round(2))
print('On the REAL S&P 500: residual skew -0.08 (vs total +0.18), drawdown -59% (vs -61%).')

,sharpe,skew,worst_month_pct,max_drawdown_pct
residual,4.4300,0.2500,-2.8000,-4.6500
total,4.0700,0.0700,-3.8500,-7.8200


On the REAL S&P 500: residual skew -0.08 (vs total +0.18), drawdown -59% (vs -61%).


> 💡 **In plain words.** Residualising flips the skew from positive to mildly negative here and lifts the alpha a touch — but a market-only residual leaves the value-driven crash (losers that are cheap, not just high-beta) on the table. That's what the FF3 residual, and vol-management, are for.

### 4c · The defence stack

In [5]:
st = extension.defence_stack(panel, market, cost_bps=5.0)
display(pd.DataFrame(st).T.round(2))
print('On the REAL S&P 500: drawdown -61% -> -59% -> -25% (Sharpe to +0.17).')

,sharpe,skew,worst_month_pct,max_drawdown_pct
total,4.0700,0.0700,-3.8500,-7.8200
residual,4.4300,0.2500,-2.8000,-4.6500
residual_vol_managed,4.3600,0.3600,-5.1400,-7.0100


On the REAL S&P 500: drawdown -61% -> -59% -> -25% (Sharpe to +0.17).


## Beat 5 · The verdict

- **Real on control** (4a): residual alpha HAC *t* ≈ 16.
- **Faint but cleaner here** (4b): alpha +6.2%/yr (*t* +1.2), skew -0.08.
- **Stack tames the tail** (4c): drawdown to -25%.

> **Signal `WEAK` · Tradability `FRAGILE` · Cleaner than total momentum? `Confirmed`.**

## Beat 6 · Could you trade it?

- **Thin standalone** (Sharpe +0.08); short-the-losers; fast turnover.
- **The right platform for crash management** — stacking vol-targeting drops the drawdown to **-25%**.
- **A 1-factor residual under-cleans** — FF3 residuals would shed more of the value-driven crash.

Tradability **`FRAGILE`**; cleaner than total `Confirmed`.

## Beat 7 · Going further

### 7a · Worked complement — the defence stack
Total-WML vs residual-WML vs vol-managed residual-WML: the progressive taming of the tail.

In [6]:
st = extension.defence_stack(panel, market, cost_bps=5.0)
for k in ['total', 'residual', 'residual_vol_managed']:
    p = st[k]; print(f"{k:22s}: Sharpe {p['sharpe']:+.2f}, skew {p['skew']:+.2f}, "
                     f"worst month {p['worst_month_pct']:+.1f}%, max drawdown {p['max_drawdown_pct']:.0f}%")
print('On the REAL S&P 500 (../docs/extension.md): drawdown -61% -> -59% -> -25%.')

total                 : Sharpe +4.07, skew +0.07, worst month -3.8%, max drawdown -8%
residual              : Sharpe +4.43, skew +0.25, worst month -2.8%, max drawdown -5%
residual_vol_managed  : Sharpe +4.36, skew +0.36, worst month -5.1%, max drawdown -7%
On the REAL S&P 500 (../docs/extension.md): drawdown -61% -> -59% -> -25%.


**The result.** Two independent crash defences compose: residualising removes the beta-driven tail, vol-scaling removes the regime-clustered remainder, and on the real S&P 500 the drawdown falls from **-61%** (raw momentum) to **-25%** while the Sharpe *rises* to **+0.17**. The crash, the thing that made [Study 24](../../24-stampede/) `FRAGILE`, is engineerable; the faint *premium* on the modern large-cap sample is the part that isn't. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **FF3 residuals** (MKT, SMB, HML) — the source's recipe; should shed the value-driven crash the 1-factor residual leaves behind.
- **Industry-/sector-neutral momentum** — an alternative neutralisation; compare the crash.
- **Constant-volatility momentum** (Barroso–Santa-Clara) applied to the residual factor.

PRs welcome — add the FF3 residual, or benchmark the neutralisations head-to-head.